构建一个包含标准数据库（如 OmniPath） + 你的自定义四元组的“基因程序字典（Gene Program Dictionary）”，然后传给模型。

In [2]:
import scanpy as sc
import numpy as np
from nichecompass.models import NicheCompass

# 1. 加载你刚才保存的数据
adata = sc.read_h5ad("/home/zhangjunyi/xiangmu/nichecompass-main/datasets/st_data/visium_lung/my_spatial_data_annotated.h5ad")

# 2. 准备原始 Counts (NicheCompass 训练必须用整数 Counts)
# 假设你的 raw counts 还在 adata.raw 或者某个 layer 里
# 如果 adata.X 已经是 log1p 的数据，请务必找到原始计数
if "counts" not in adata.layers:
    # 这是一个假设，通常读取进来如果没有 counts 层，你需要确认一下 adata.X 是不是整数
    # 如果不是，你可能需要回溯一步把 counts 存进去
    # 这里假设你之前处理时没有丢掉 raw 数据
    adata.layers["counts"] = adata.raw.X.copy() if adata.raw is not None else adata.X.copy()

# --- 关键步骤：定义你的自定义四元组 ---
# 将你的四元组定义为字典。键是名字，值是基因列表。
# 比如你有两组感兴趣的四元组
my_custom_gps = {
    "My_Hypothesis_1": ["GeneA", "Ligand1", "Receptor1", "Target1"], 
    "My_Hypothesis_2": ["GeneB", "Ligand2", "Receptor2", "Target2"]
}

print("自定义四元组准备完毕。")

自定义四元组准备完毕。


步骤 2：获取 NicheCompass 的默认先验库并合并
为了保证准确性（即你说的“按照 NicheCompass 的流程走”），我们需要加载它内置的强大的配体-受体库（OmniPath），然后把你的加进去。

In [3]:
from nichecompass.utils import extract_gp_dict_from_omics_matrix, get_unique_genes_from_gp_dict

# 1. 这里我们需要加载 NicheCompass 内置的 GP 库
# 注意：通常 NicheCompass 会自动下载或构建，但为了插入自定义，我们需要手动处理一下
# 如果你是初次运行，最简单的方法是让模型帮我们构建默认的，然后我们再添加

# 由于 NicheCompass API 的特性，我们通常在模型初始化时传入 `gp_names_key`
# 但为了强制加入你的基因，我们最好显式地构建这个字典。

# 假设我们先使用 NicheCompass 默认的方法获取 GP (这里简化处理，通常包里有 fetch 函数)
# 为了不破坏流程，我们采用 "Add-on" 策略：

# NicheCompass 允许在 adata.varm 中存储 GP mask
# 但更简单的做法是在训练配置时指定。

# --- 修正策略：直接在训练前将你的基因标记为“高关注” ---

# 实际上，NicheCompass 会自动从数据库学习。
# 如果你的四元组基因已经在 OmniPath 数据库里，模型会自动捕捉。
# 如果不在（或者是你特有的组合），你需要这样做：

# 检查你的基因是否在 adata 的 var_names 里
for gp_name, genes in my_custom_gps.items():
    missing = [g for g in genes if g not in adata.var_names]
    if missing:
        print(f"警告：在数据中找不到以下基因: {missing}，它们将被忽略。")

ImportError: cannot import name 'extract_gp_dict_from_omics_matrix' from 'nichecompass.utils' (/home/zhangjunyi/anaconda3/envs/NicheCompass/lib/python3.9/site-packages/nichecompass/utils/__init__.py)

第一步：读取数据并准备 NicheCompass 环境

In [1]:
import scanpy as sc
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# 1. 加载你刚才保存的文件
file_path = "/home/zhangjunyi/xiangmu/nichecompass-main/datasets/st_data/visium_lung/my_spatial_data_annotated.h5ad"
adata = sc.read_h5ad(file_path)

print(f"数据加载成功，包含 {adata.n_obs} 个 Spots 和 {adata.n_vars} 个基因")

# [关键检查] NicheCompass 推荐使用原始 Counts (整数) 进行训练
# 如果 adata.X 已经是 log 后的数据，我们需要把 raw counts 找回来放在 layer 里
# 假设 adata.raw 或 adata.layers['counts'] 里有，如果没有，请确保 adata.X 是整数
# 这里为了保险，我们检查一下，如果 X 是浮点数且没有 counts 层，你可能需要重新从原始文件读一遍 counts
if 'counts' not in adata.layers:
    # 简单的尝试：如果你之前的 adata.X 没被覆盖，或者有备份
    # 这里假设 adata.X 已经是 log-normalized 了，通常预处理时我们会把 counts 存一份
    # 如果你确定 adata.X 是原始计数，直接：adata.layers['counts'] = adata.X.copy()
    pass

数据加载成功，包含 3816 个 Spots 和 18039 个基因


第二步：构建空间邻接图 (Spatial Graph)
NicheCompass 需要知道哪些 Spot 是相邻的，以便学习空间依赖关系

In [ ]:
from nichecompass.utils import compute_spatial_neighbors

# 2. 计算空间邻居图
# 对于 Visium 数据，neighbors 通常设为 6 (六边形)
# n_rings=1 表示只看直接相邻的一圈
compute_spatial_neighbors(
    adata,
    spatial_key="spatial",   # 你的坐标列名
    coord_type="generic",    # Visium 可以用 grid 或 generic
    n_neighbors=6,
    n_rings=1
)

print("空间邻接图构建完成。")